# 실험 E1 — Qwen3.5-9B 단독 (이미지, CoT 없음)

**목적:** 모델 크기의 효과만 분리해 측정합니다. 실험 B(Qwen2.5-VL-3B, 0.9033)와 **입력·프롬프트·토큰 한도가 동일**하고 모델만 다릅니다.

`E1 − B` = 3B → 9B 교체의 효과.

## 설계상 주의

- **OCR을 쓰지 않습니다.** 실험 C(OCR+VLM, 0.8973)가 B(0.9033)와 통계적으로 구분되지 않아(McNemar p=0.557) OCR 단계를 뺐습니다
- **9B는 4bit NF4입니다.** bf16이면 18.0 GiB로 16 GiB GPU에 올라가지 않습니다.
  B는 bf16이었으므로 `E1 − B`에는 **모델 크기와 양자화 차이가 섞여 있습니다.** 결과 해석 시 명시하세요
- 이미지 규칙(긴 변 960px), 프롬프트, 파서, 검증 672문항은 B와 동일합니다

In [ ]:
# Resolve shared code and local assets from the repository root.
import os as _project_os
import sys as _project_sys
from pathlib import Path as _ProjectPath
_project_root = next((p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                      if (p / 'src' / 'ocr_engine.py').is_file()), None)
if _project_root is not None:
    _project_os.chdir(_project_root)
    if str(_project_root / 'src') not in _project_sys.path:
        _project_sys.path.insert(0, str(_project_root / 'src'))
# End repository path setup.

import gc, json, os, time
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm

import common_eval as ce

VLM_MODEL_DIR = Path("downloads/models/Qwen3.5-9B")
OUTPUT_DIR = Path("output/exp_E1_qwen35_9b")
MAX_NEW_TOKENS = 24         # B/C/D와 동일
VLM_4BIT = True            # 9B bf16은 18.0 GiB라 16 GiB GPU에 올라가지 않습니다
ENABLE_THINKING = False    # Qwen3.5 자체 thinking 모드. CoT는 프롬프트로만 유도합니다
SMOKE_TEST = False         # True면 앞 8문항만

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ[key] = "1"

assert torch.cuda.is_available(), "이 실험 설정은 CUDA GPU를 사용합니다."
assert (VLM_MODEL_DIR / "config.json").is_file(), f"로컬 모델을 준비하세요: {VLM_MODEL_DIR}"
shards = sorted(VLM_MODEL_DIR.glob("*.safetensors"))
index = VLM_MODEL_DIR / "model.safetensors.index.json"
if index.is_file():
    need = sorted(set(json.loads(index.read_text())["weight_map"].values()))
    miss = [f for f in need if not (VLM_MODEL_DIR / f).is_file()]
    assert not miss, f"가중치 shard {len(miss)}/{len(need)}개 누락. 다운로드를 끝내세요: {miss[:2]}"
assert shards, f"가중치(.safetensors)가 없습니다: {VLM_MODEL_DIR}"

DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(42)
ce._self_test()
print("GPU:", torch.cuda.get_device_name(0), "/ dtype:", DTYPE)
print("shard", len(shards), "개 / 4bit:", VLM_4BIT)
print("이미지 긴 변 제한:", ce.IMAGE_MAX_EDGE, "px (실험 B/C와 동일)")

## 검증 분할
실험 A가 저장한 CSV를 그대로 읽고 ID 해시로 동일 분할임을 확인합니다.

In [ ]:
val_df = ce.load_val()
if SMOKE_TEST:
    print("⚠ SMOKE_TEST: 공식 비교 결과로 사용 금지")
    val_df = val_df.head(8).copy()
rows = val_df.drop(columns=["answer"]).to_dict("records")   # 정답은 입력에서 제거
print("추론 대상:", len(rows))

## 프롬프트
`ce.make_prompt`와 `ce.SYSTEM`을 그대로 씁니다 — 실험 B와 **글자 단위로 동일**합니다.

In [ ]:
def build_messages(row, image):
    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},
            {"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": ce.make_prompt(row)}]}]

# 셀 3을 건너뛰고 제출 추론만 할 수도 있으므로, 없으면 예시 문항으로 미리 봅니다.
try:
    _preview_row = rows[0]
except NameError:
    _preview_row = {"question": "(예시) 이 간판의 상호명은 무엇인가요?",
                    "a": "가게1", "b": "가게2", "c": "가게3", "d": "가게4"}
print(build_messages(_preview_row, None)[1]["content"][1]["text"])

## 모델 로드
9B를 4bit로 올립니다. 가중치 shard 4개(총 18 GiB)가 모두 있어야 합니다.

In [ ]:
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

def load_vlm():
    proc = AutoProcessor.from_pretrained(str(VLM_MODEL_DIR), local_files_only=True)
    kwargs = dict(local_files_only=True, dtype=DTYPE, device_map={"": 0})
    if VLM_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(VLM_MODEL_DIR), **kwargs)
    return model.eval(), proc

@torch.inference_mode()
def generate_text(model, proc, inputs, limit):
    inputs = inputs.to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    output = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True)
    new = output[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated), int(len(new))

## 추론

In [ ]:
model, proc = load_vlm()
torch.cuda.reset_peak_memory_stats()
results = []
try:
    with (OUTPUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as log:
        for row in tqdm(rows, desc="E1: 9B (CoT 없음)"):
            torch.cuda.synchronize(); start = time.perf_counter()
            im = ce.open_image(row)
            inputs = proc.apply_chat_template(
                build_messages(row, im), add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", enable_thinking=ENABLE_THINKING)
            raw, capped, n_tok = generate_text(model, proc, inputs, MAX_NEW_TOKENS)
            torch.cuda.synchronize(); elapsed = time.perf_counter() - start
            item = dict(id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),
                        answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,
                        image_width=im.width, image_height=im.height)
            results.append(item)
            log.write(json.dumps(item, ensure_ascii=False) + "\n"); log.flush()
            del inputs, im
    peak = torch.cuda.max_memory_allocated() / 1024**3
finally:
    del model, proc
    gc.collect(); torch.cuda.empty_cache()
print(f"완료 / GPU peak {peak:.2f} GiB")

## 분석

In [ ]:
pred_df = val_df.merge(pd.DataFrame(results), on="id", validate="one_to_one", sort=False)
assert pred_df.id.tolist() == val_df.id.tolist()
pred_df["correct"] = pred_df.pred == pred_df.answer

score = ce.save_analysis(pred_df, "E1_qwen35_9b", OUTPUT_DIR, extra=dict(
    vlm_model=str(VLM_MODEL_DIR), vlm_4bit=VLM_4BIT, image_max_edge=ce.IMAGE_MAX_EDGE,
    answer_max_new_tokens=MAX_NEW_TOKENS, enable_thinking=ENABLE_THINKING,
    peak_allocated_gib=peak, cot=False,
    answer_truncated_rate=float(pred_df.answer_truncated.mean()),
    mean_answer_tokens=float(pred_df.answer_tokens.mean()),
    uses_image=True, uses_ocr=False, smoke=SMOKE_TEST))

# 예측이 한쪽으로 쏠리면 모델이 읽지 않고 찍고 있다는 신호입니다.
print("\n예측 분포 (정답 분포와 비슷해야 정상)")
comp = pd.DataFrame({"정답": val_df.answer.value_counts().reindex(ce.CHOICES),
                     "예측": pred_df.pred.value_counts().reindex(ce.CHOICES)}).fillna(0).astype(int)
print(comp.to_string())
print(f"\nB(Qwen2.5-VL-3B 단독) = 0.9033 과 비교: {score['accuracy'] - 0.9033:+.4f}")
print("산출물:", OUTPUT_DIR.resolve())

---

## 제출 추론 (test.csv)

위 검증 셀과 **완전히 같은 프롬프트·이미지 규칙·모델 설정**으로 추론해 제출 파일을 만듭니다.
설정을 공유하므로 검증 정확도와 제출 점수를 같은 조건에서 비교할 수 있습니다.

**실행 순서:** 셀 1 -> 5 -> 7 -> 아래 셀들.

위쪽 **검증 섹션(셀 3·9·11)은 실행하지 않아도 됩니다.** 그 셀들은 실험 A가 저장한 672문항 분할을
고정으로 쓰는 비교용이라 여기서 대상을 바꿔도 영향을 주지 않습니다.
전체 데이터를 돌리는 것은 아래 `SUBMISSION_CSV`이고, 둘은 서로 독립입니다.
셀 9를 이미 돌렸다면 모델이 해제된 상태라 아래에서 다시 올립니다.

검증과 다른 점은 두 가지입니다.

- 정답이 없으므로 채점 대신 `submission.csv`를 만듭니다
- 파서가 실패한 문항을 **빈칸이나 고정 글자로 두지 않고** a~d 로짓을 비교해 채웁니다
  (기존 베이스라인 `submission.csv`가 `a` 42%로 쏠린 원인이 `return "a"` 폴백이었습니다)

`SUBMISSION_CSV`를 정답이 있는 CSV로 바꾸면 그대로 채점 모드로 동작합니다. 맨 아래 가이드를 참고하세요.

In [ ]:
# --- 제출 설정 (여기만 바꾸면 됩니다) --------------------------------------
SUBMISSION_CSV     = Path("data/test.csv")               # 추론 대상
SAMPLE_SUBMISSION  = Path("data/sample_submission.csv")  # id 순서 기준. test가 아니면 무시
SUBMISSION_DIR     = None        # None이면 대상 CSV 이름으로 자동: output/exp_E1_<이름>
SUBMISSION_NAME    = None        # 이름을 직접 정할 때. None이면 아래 두 값으로 자동 생성
SUBMISSION_METHOD  = "zeroshot"  # 방식 태그. CoT면 "cot", OCR을 끼우면 "ocr-vlm"
SUBMISSION_VERSION = None        # None이면 같은 모델·방식의 다음 번호를 자동으로 붙입니다
SUBMISSION_LIMIT   = None        # int면 앞 N개만 추론(배관 점검용). 제출 파일은 만들지 않습니다
RESUME             = True        # predictions.jsonl에 남은 문항을 건너뛰고 이어서 진행
SEC_PER_ITEM       = 0.805       # E1 검증 실측(672문항 540.8초). 남은 시간 추정에만 씁니다
# ---------------------------------------------------------------------------

# 대상만 바꾸면 출력 폴더도 따라 바뀌므로 둘을 함께 고칠 일이 없습니다.
# test.csv -> output/exp_E1_test, train.csv -> output/exp_E1_train
if SUBMISSION_DIR is None:
    SUBMISSION_DIR = Path("output") / f"exp_E1_{SUBMISSION_CSV.stem}"
print("출력 폴더:", SUBMISSION_DIR)

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = SUBMISSION_DIR / "predictions.jsonl"
META_PATH = SUBMISSION_DIR / "run_meta.json"

# 이 폴더의 예측이 어떤 조건으로 만들어졌는지 기록해 둡니다. 재개할 때 대조에 씁니다.
RUN_META = {"source_csv": str(SUBMISSION_CSV), "model": str(VLM_MODEL_DIR),
            "vlm_4bit": VLM_4BIT, "max_new_tokens": MAX_NEW_TOKENS,
            "enable_thinking": ENABLE_THINKING, "image_max_edge": ce.IMAGE_MAX_EDGE}

# B/C/D/E1처럼 이미 끝난 실험 폴더를 가리키면 그 결과를 덮어쓰게 되므로 먼저 막습니다.
assert not ((SUBMISSION_DIR / "metrics.json").is_file() and not META_PATH.is_file()), (
    f"{SUBMISSION_DIR}에 다른 실험의 결과가 있습니다. "
    "SUBMISSION_DIR를 새 경로로 바꾸세요")

target_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
REQUIRED = ["id", "path", "question", "a", "b", "c", "d"]
missing_cols = sorted(set(REQUIRED) - set(target_df.columns))
assert not missing_cols, f"CSV 누락 컬럼: {missing_cols}"
assert target_df["id"].is_unique, "id 중복이 있습니다"
assert (target_df["path"].str.strip() != "").all(), "빈 이미지 경로가 있습니다"

missing_img = [p for p in target_df.path if not (ce.DATA_DIR / str(p)).is_file()]
assert not missing_img, f"이미지 누락 {len(missing_img)}개. 예: {missing_img[:3]}"

# 정답 컬럼이 있으면(train 등) 채점용으로 따로 빼두고, 모델 입력에서는 반드시 제거합니다.
if "answer" in target_df.columns:
    GOLD = target_df.set_index("id")["answer"].astype(str).str.strip().str.lower()
    assert GOLD.isin(ce.CHOICES).all(), "a~d 이외의 정답이 있습니다"
    print(f"정답 컬럼 발견 -> 채점 모드 ({len(GOLD)}개)")
else:
    GOLD = None
    print("정답 컬럼 없음 -> 제출 전용 모드")

IS_TEST_SPLIT = SUBMISSION_CSV.name == "test.csv"
sample_df = pd.read_csv(SAMPLE_SUBMISSION, keep_default_na=False) if IS_TEST_SPLIT else None
if sample_df is not None:
    assert list(target_df.id) == list(sample_df.id), \
        "test.csv와 sample_submission.csv의 id 순서가 다릅니다. 제출 형식이 깨집니다"

target_rows = target_df.drop(columns=["answer"], errors="ignore").to_dict("records")
if SUBMISSION_LIMIT:
    target_rows = target_rows[:SUBMISSION_LIMIT]
    print(f"SUBMISSION_LIMIT={SUBMISSION_LIMIT} -> 제출 파일은 만들지 않습니다")

done_ids = set()
if RESUME and PRED_PATH.is_file():
    # 대상 CSV나 모델을 바꾼 채 이어 붙이면 서로 다른 조건의 예측이 한 파일에 섞입니다.
    if META_PATH.is_file():
        prev_meta = json.loads(META_PATH.read_text(encoding="utf-8"))
        changed = {k: f"{prev_meta.get(k)} -> {v}"
                   for k, v in RUN_META.items() if prev_meta.get(k) != v}
        assert not changed, (
            f"기존 예측과 실행 조건이 다릅니다: {changed}. "
            "SUBMISSION_DIR를 새 경로로 지정하거나 RESUME=False로 두세요")
    with PRED_PATH.open(encoding="utf-8") as f:
        done_ids = {json.loads(line)["id"] for line in f if line.strip()}
    print(f"이어서 진행: 이미 끝난 {len(done_ids)}개 건너뜀")
elif not RESUME and PRED_PATH.is_file():
    PRED_PATH.unlink()
    print("RESUME=False -> 기존 predictions.jsonl 삭제")

META_PATH.write_text(json.dumps(RUN_META, ensure_ascii=False, indent=2), encoding="utf-8")

todo = [r for r in target_rows if r["id"] not in done_ids]
print(f"추론 대상 {len(todo)}개 / 남은 예상 시간 약 {len(todo) * SEC_PER_ITEM / 60:.0f}분")

### 폴백 — 파싱 실패를 고정 글자로 채우지 않기

`ce.parse_choice`는 출력이 정확히 한 글자일 때만 인정하므로 나머지는 `INVALID`가 됩니다.
제출 파일에는 빈칸을 둘 수 없는데, 특정 글자로 메우면 그 글자만 과대 예측되어 분포가 무너집니다.
대신 **다음 토큰 분포에서 a~d만 비교**해 가장 높은 것을 고릅니다. 추가 비용은 forward 1회입니다.

In [ ]:
def letter_token_ids(proc):
    """a~d를 시작하는 첫 토큰 후보. 토크나이저마다 선행 공백/대소문자 처리가 달라 모두 모읍니다."""
    tok = proc.tokenizer
    table = {}
    for ch in ce.CHOICES:
        cand = set()
        for form in (ch, " " + ch, ch.upper(), " " + ch.upper()):
            enc = tok.encode(form, add_special_tokens=False)
            if enc:
                cand.add(int(enc[0]))
        assert cand, f"'{ch}'의 토큰 id를 찾지 못했습니다"
        table[ch] = sorted(cand)
    return table


@torch.inference_mode()
def choose_by_logit(model, proc, inputs, table):
    """생성 결과가 INVALID일 때만 호출하는 폴백. a~d 중 로짓이 가장 높은 글자를 돌려줍니다."""
    inputs = inputs.to(model.device)
    logits = model(**inputs).logits[0, -1].float()
    scores = {ch: max(logits[i].item() for i in ids) for ch, ids in table.items()}
    return max(scores, key=scores.get), scores


### 추론 루프

한 건 끝날 때마다 `predictions.jsonl`에 바로 쓰고 flush하므로, 중단되어도 이 셀만 다시 실행하면
`RESUME=True` 기준으로 남은 문항부터 이어집니다.

In [ ]:
model, proc = load_vlm()
LETTERS = letter_token_ids(proc)
torch.cuda.reset_peak_memory_stats()
n_fallback = 0
peak = float("nan")
try:
    with PRED_PATH.open("a", encoding="utf-8") as log:
        for row in tqdm(todo, desc="E1 제출 추론"):
            torch.cuda.synchronize(); start = time.perf_counter()
            im = ce.open_image(row)
            inputs = proc.apply_chat_template(
                build_messages(row, im), add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", enable_thinking=ENABLE_THINKING)
            raw, capped, n_tok = generate_text(model, proc, inputs, MAX_NEW_TOKENS)
            pred = ce.parse_choice(raw)
            fallback = pred == "INVALID"
            if fallback:
                pred, _ = choose_by_logit(model, proc, inputs, LETTERS)
                n_fallback += 1
            torch.cuda.synchronize(); elapsed = time.perf_counter() - start
            log.write(json.dumps(dict(
                id=row["id"], raw_output=raw, pred=pred, fallback=fallback,
                answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,
                image_width=im.width, image_height=im.height), ensure_ascii=False) + "\n")
            log.flush()
            del inputs, im
    peak = torch.cuda.max_memory_allocated() / 1024**3
finally:
    del model, proc
    gc.collect(); torch.cuda.empty_cache()

rate = n_fallback / max(len(todo), 1)
print(f"완료 / 로짓 폴백 {n_fallback}개 ({rate:.2%}) / GPU peak {peak:.2f} GiB")
if rate > 0.02:
    print("경고: 폴백 비율이 높습니다. MAX_NEW_TOKENS를 늘리거나 출력 형식을 점검하세요")

### 제출 파일 생성과 검증

저장 전에 행 수·id 순서·값 범위를 확인하고, 예측 분포가 한쪽으로 쏠리면 경고합니다.
train 정답 분포의 최대 비중이 25.6%이므로 35%를 넘으면 파이프라인을 의심해야 합니다.

In [ ]:
import re


def submission_filename():
    """제출 파일 이름: submission_<모델>_<방식>_v<버전>.csv

    예: submission_qwen35-9b_zeroshot_v1.csv

    SUBMISSION_NAME을 주면 그대로 씁니다.
    SUBMISSION_VERSION이 None이면 같은 모델·방식으로 이미 만든 파일을 보고 다음 번호를 붙이므로,
    여러 번 제출해도 이전 파일을 덮어쓰지 않습니다.
    """
    if SUBMISSION_NAME:
        name = str(SUBMISSION_NAME)
        return name if name.endswith(".csv") else name + ".csv"

    model_tag = VLM_MODEL_DIR.name.replace(".", "").lower()   # Qwen3.5-9B -> qwen35-9b
    stem = f"submission_{model_tag}_{SUBMISSION_METHOD}"

    version = SUBMISSION_VERSION
    if version is None:
        pattern = re.compile(re.escape(stem) + r"_v(\d+)$")
        used = []
        for path in SUBMISSION_DIR.glob(stem + "_v*.csv"):
            matched = pattern.fullmatch(path.stem)
            if matched:
                used.append(int(matched.group(1)))
        version = max(used, default=0) + 1
    return f"{stem}_v{version}.csv"


assert PRED_PATH.is_file(), "예측 파일이 없습니다. 추론 셀을 먼저 실행하세요"
pred_df = pd.read_json(PRED_PATH, lines=True).drop_duplicates("id", keep="last")
print(f"기록된 예측 {len(pred_df)}개 / 대상 {len(target_rows)}개")

if SUBMISSION_LIMIT:
    print("SUBMISSION_LIMIT이 설정되어 제출 파일을 만들지 않습니다.")
else:
    assert len(pred_df) >= len(target_df), \
        f"예측 부족 {len(pred_df)}/{len(target_df)}. 추론 셀을 다시 실행해 이어서 채우세요"
    sub = target_df[["id"]].merge(pred_df[["id", "pred"]], on="id",
                                  how="left", validate="one_to_one")
    assert sub.pred.notna().all(), "비어 있는 예측이 있습니다"
    assert sub.pred.isin(ce.CHOICES).all(), \
        f"a~d 이외의 값: {sorted(set(sub.pred) - set(ce.CHOICES))}"
    if sample_df is not None:
        assert list(sub.id) == list(sample_df.id), "id 순서가 sample_submission과 다릅니다"

    if IS_TEST_SPLIT:
        out_path = SUBMISSION_DIR / submission_filename()
        assert not out_path.exists(), (
            f"같은 이름의 제출 파일이 이미 있습니다: {out_path.name}. "
            "SUBMISSION_NAME을 바꾸거나 기존 파일을 옮기세요")
        sub.rename(columns={"pred": "answer"}).to_csv(out_path, index=False,
                                                      encoding="utf-8")
    else:
        # train 등으로 돌린 결과를 실수로 제출하지 않도록 파일을 만들지 않습니다.
        out_path = None
        print("test.csv가 아니므로 제출 파일은 만들지 않습니다 (분포 점검·채점만 수행)")

    counts = sub.pred.value_counts().reindex(ce.CHOICES).fillna(0).astype(int)
    share = counts / counts.sum()
    print("\n예측 분포")
    print(pd.DataFrame({"건수": counts, "비중": share.round(3)}).to_string())
    if share.max() > 0.35:
        print(f"\n경고: '{share.idxmax()}'가 {share.max():.1%}로 쏠렸습니다. "
              "train 정답 분포의 최대 비중은 25.6%입니다. 파서와 폴백을 점검하세요")
    else:
        print(f"\n분포 정상 (최대 {share.max():.1%})")
    if out_path is not None:
        print("제출 파일:", out_path.resolve())

### 선택 — 정답이 있는 CSV로 돌렸을 때 채점

`SUBMISSION_CSV`가 `train.csv`처럼 정답을 가진 파일이면 아래 셀이 검증과 같은 지표
(정확도·macro-F1·혼동행렬)를 남깁니다. 정답이 없으면 그냥 건너뜁니다.

In [ ]:
if GOLD is None:
    print("정답이 없는 CSV입니다. 채점을 건너뜁니다.")
else:
    scored = pred_df[["id", "pred", "seconds"]].copy()
    scored["answer"] = scored.id.map(GOLD)
    scored = scored.dropna(subset=["answer"])
    scored["correct"] = scored.pred == scored.answer

    full_score = ce.save_analysis(scored, f"E1_{SUBMISSION_CSV.stem}", SUBMISSION_DIR, extra=dict(
        vlm_model=str(VLM_MODEL_DIR), vlm_4bit=VLM_4BIT, image_max_edge=ce.IMAGE_MAX_EDGE,
        answer_max_new_tokens=MAX_NEW_TOKENS, enable_thinking=ENABLE_THINKING, cot=False,
        source_csv=str(SUBMISSION_CSV), logit_fallbacks=int(pred_df.fallback.sum()),
        uses_image=True, uses_ocr=False))

    # 표본이 커지면 정확도가 오르는 게 아니라 신뢰구간이 좁아집니다. 그 폭을 같이 남깁니다.
    from math import sqrt
    k, n, z = int(scored.correct.sum()), len(scored), 1.96
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    print(f"\n정확도 {p:.4f} ({k}/{n}) / 95% CI [{centre - half:.4f}, {centre + half:.4f}] "
          f"= +-{half * 100:.2f}pp")
    print(f"검증 672문항(0.9420)과 비교: {p - 0.9420:+.4f}")

---

## 사용 가이드 — 직접 고쳐 쓸 때

### 무엇을 바꾸면 되나

| 값 | 위치 | 언제 바꾸나 |
| --- | --- | --- |
| `SUBMISSION_CSV` | 제출 설정 셀 | 대상 변경. `data/test.csv`(제출) ↔ `data/train.csv`(전체 채점) |
| `SUBMISSION_DIR` | 제출 설정 셀 | 보통 건드릴 일이 없습니다. `None`이면 `output/exp_E1_<대상이름>`으로 자동 결정됩니다 |
| `SUBMISSION_NAME` | 제출 설정 셀 | 이름을 통째로 고정할 때. `None`이면 `submission_<모델>_<방식>_v<버전>.csv`로 자동 생성 |
| `SUBMISSION_METHOD` | 제출 설정 셀 | 파일명의 방식 태그. 기본 `"zeroshot"`, CoT로 돌리면 `"cot"`으로 바꾸세요 |
| `SUBMISSION_VERSION` | 제출 설정 셀 | 회차 번호. `None`이면 같은 모델·방식의 기존 파일을 보고 다음 번호를 자동으로 붙입니다 |
| `SUBMISSION_LIMIT` | 제출 설정 셀 | 배관 점검. `8`로 두고 한 번 돌려 오류를 먼저 걸러내세요 |
| `RESUME` | 제출 설정 셀 | `False`면 기존 예측을 지우고 처음부터. 설정을 바꿨다면 반드시 `False` |
| `VLM_MODEL_DIR` | 셀 1 | 다른 모델과 비교할 때 |
| `VLM_4BIT` | 셀 1 | VRAM이 넉넉하면 `False`(bf16). 9B는 bf16이 18.0 GiB라 16 GiB GPU에서는 불가 |
| `MAX_NEW_TOKENS` | 셀 1 | 폴백 비율이 높을 때만 상향. CoT로 바꾸면 최소 256 필요 |
| `ENABLE_THINKING` | 셀 1 | Qwen3.5 자체 thinking 모드. 켜면 출력이 길어져 `parse_choice`가 전부 실패합니다 |

### 바꾸면 비교가 깨지는 것

`ce.make_prompt` / `ce.SYSTEM` / `ce.IMAGE_MAX_EDGE` / `ce.parse_choice`는
실험 A~E가 공유합니다. 여기서 고치면 B·C·D·E1 결과와 더 이상 비교할 수 없습니다.
바꿔야 한다면 [src/common_eval.py](../../src/common_eval.py)에서 고치고 **모든 실험을 다시 돌려야** 합니다.

### CoT로 바꾸려면

이 노트북을 고치지 말고 [E2 노트북](AI_Challenge2_expE2_qwen35_9b_cot.ipynb)에 같은 방식으로
제출 셀을 붙이세요. CoT는 네 군데를 함께 바꿔야 합니다.

- `ce.SYSTEM` -> `ce.SYSTEM_COT`
- `ce.make_prompt(row)` -> `ce.make_prompt(row, cot=True)`
- `ce.parse_choice` -> `ce.parse_choice_cot`
- `MAX_NEW_TOKENS` 24 -> 256 이상

네 개 중 하나라도 빠지면 출력이 전부 `INVALID`가 되어 폴백만 돌아갑니다.

### 검증 672문항과 전체 데이터의 관계

셀 3의 `ce.load_val()`은 실험 A가 저장한 **672문항 고정 분할**입니다. B·C·D·E1을 같은 문항으로
비교하기 위한 것이라 숫자를 바꾸면 안 되고, 바꿀 필요도 없습니다.
전체 데이터 실행은 제출 설정의 `SUBMISSION_CSV`가 담당하며 두 경로는 서로 영향을 주지 않습니다.

### train 전체를 채점하려면

```python
SUBMISSION_CSV = Path("data/train.csv")
```

이 한 줄이면 됩니다. 출력은 `output/exp_E1_train/`으로 자동 분리됩니다.

E1은 학습을 하지 않으므로 train 6,714개 전부를 평가에 써도 누수가 없습니다.
다만 672개(94.20%)는 이미 층화 무작위 추출이라 **정확도가 올라가지는 않습니다.**
신뢰구간이 ±1.78pp에서 ±0.56pp로 좁아질 뿐입니다.

### 기존 결과를 덮어쓰지 않는가

이 섹션은 `SUBMISSION_DIR`(기본 `output/exp_E1_submission`) **안에만** 씁니다.
672문항 검증 결과인 `output/exp_E1_qwen35_9b/`와 B·C·D 결과 폴더는 건드리지 않습니다.

폴더를 잘못 지정했을 때를 대비해 두 가지를 막아 둡니다.

- `metrics.json`은 있는데 `run_meta.json`이 없는 폴더 → 다른 실험 결과로 보고 **실행 자체를 중단**
- 같은 폴더에 조건이 다른 예측이 있을 때(`RESUME=True`) → `run_meta.json`과 대조해 중단.
  대상 CSV·모델·양자화·토큰 한도·thinking·이미지 크기가 하나라도 다르면 걸립니다

`SUBMISSION_DIR`이 `None`이면 대상 CSV 이름으로 폴더가 갈리므로
(`output/exp_E1_test`, `output/exp_E1_train`) 평소에는 위 장치가 걸릴 일이 없습니다.
폴더를 직접 지정했을 때를 위한 안전망입니다.

### 중단·재개와 OOM

- 중단되면 추론 셀만 다시 실행하세요. `RESUME=True`면 남은 문항부터 이어집니다
- OOM이면 `ce.IMAGE_MAX_EDGE`를 낮추기 전에 `VLM_4BIT=True`인지부터 확인하세요.
  `IMAGE_MAX_EDGE`를 바꾸면 B·C와 비교가 깨집니다
- 커널을 재시작했다면 셀 1 -> 5 -> 7을 다시 실행한 뒤 제출 셀로 오세요

### 제출 전 점검표

1. 행 수가 6,714이고 id 순서가 `sample_submission.csv`와 같은가 (생성 셀이 assert로 확인)
2. 예측 분포의 최대 비중이 35% 미만인가 — 넘으면 파서나 폴백이 고장난 신호
3. 로짓 폴백 비율이 2% 미만인가
4. `metrics.json` 또는 로그에 사용한 모델·설정이 남아 있는가

**폴백을 특정 글자로 되돌리지 마세요.** 기존 베이스라인이 `return "a"`를 써서
제출 분포가 `a` 42%(2,804/6,714)로 무너졌습니다. train 정답 분포는 네 선지가 거의 균등합니다.